<a href="https://colab.research.google.com/github/smir-nat/IAD-Practicals/blob/main/lab2/lab_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Завдання 1

In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, f1_score

# 1. Завантаження та підготовка даних (як у Лаб 1)
# Якщо файл лежить на GitHub, встав сюди своє посилання (Raw URL), або просто назву файлу
df = pd.read_csv('cybersecurity_lab1.csv')

cols_to_drop = ['timestamp', 'src_ip', 'dst_ip', 'url', 'attack_type']
df_clean = df.drop(columns=cols_to_drop)

le = LabelEncoder()
df_clean['protocol'] = le.fit_transform(df_clean['protocol'])
df_clean['user_agent'] = le.fit_transform(df_clean['user_agent'])
df_clean['is_internal_traffic'] = df_clean['is_internal_traffic'].astype(int)

X = df_clean.drop(columns=['label'])
y = df_clean['label']

# Розбиття та масштабування (StandardScaler обов'язковий для PCA та t-SNE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Кількість ознак ДО зниження розмірності: {X_train.shape[1]}")

# ==========================================
# 2. Базова класифікація ДО застосування PCA
# ==========================================
dt_base = DecisionTreeClassifier(random_state=42)
start_time = time.time()
dt_base.fit(X_train_scaled, y_train)
time_base = time.time() - start_time
y_pred_base = dt_base.predict(X_test_scaled)
f1_base = f1_score(y_test, y_pred_base)

# ==========================================
# 3. Зниження розмірності через PCA
# ==========================================
# Залишаємо компоненти, які пояснюють 95% дисперсії (інформації)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Кількість ознак ПІСЛЯ PCA: {X_train_pca.shape[1]}")

# Класифікація ПІСЛЯ застосування PCA
dt_pca = DecisionTreeClassifier(random_state=42)
start_time = time.time()
dt_pca.fit(X_train_pca, y_train)
time_pca = time.time() - start_time
y_pred_pca = dt_pca.predict(X_test_pca)
f1_pca = f1_score(y_test, y_pred_pca)

print("\n--- Порівняння результатів ---")
print(f"ДО PCA   | F1-Score: {f1_base:.4f} | Час навчання: {time_base:.4f} сек")
print(f"ПІСЛЯ PCA| F1-Score: {f1_pca:.4f} | Час навчання: {time_pca:.4f} сек")

# ==========================================
# 4. Візуалізація даних у 2D за допомогою t-SNE
# ==========================================
print("\nОбчислення t-SNE (це може зайняти 10-20 секунд)...")
# t-SNE працює довго, тому візьмемо лише тренувальну вибірку
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_train_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=X_tsne[:, 0], y=X_tsne[:, 1],
    hue=y_train,
    palette={0: 'lightsteelblue', 1: 'crimson'},
    alpha=0.7, s=30
)
plt.title('Візуалізація мережевого трафіку у 2D (t-SNE)', fontsize=14)
plt.xlabel('t-SNE компонент 1')
plt.ylabel('t-SNE компонент 2')
# Змінюємо назви в легенді для краси
plt.legend(title='Клас', labels=['0 (Нормальний)', '1 (Атака)'])
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'cybersecurity_lab1.csv'